Evaluation of instruction fine-tuned Phi-1.5 model

prompt engineering evaluation + measuring inference time

In [5]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_PATH = 'my_phi_model'
phi = AutoModelForCausalLM.from_pretrained(MODEL_PATH, trust_remote_code=True, device_map="auto" if torch.cuda.is_available() else None)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

In [28]:
import time
import pandas as pd

def generate_text(model, tokenizer, prompt, max_new_tokens=150):
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, repetition_penalty=1.2, no_repeat_ngram_size=3)
    output_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    text = tokenizer.decode(output_tokens, skip_special_tokens=True)
    return text.strip()

def experiment(prompts):
    results = []
    for prompt in prompts:
        print(f"Prompt: {prompt}")
        # measuring inference time
        start = time.time()
        resp = generate_text(phi, tokenizer, prompt)
        elapsed = time.time() - start
        resp_len = len(resp.split())
        # counting presence of important words
        keywords = ["food", "housing", "exercise", "health", "social", "clean", "enclosure", "outside", "care"]
        match_count = sum(1 for w in keywords if w in resp.lower())
        results.append({
            "prompt": prompt,
            "response": resp,
            "words": resp_len,
            "time_s": round(elapsed, 2),
            "keyword_matches": match_count
        })
        print(f"Response: {resp}")
    
    df = pd.DataFrame(results).sort_values("prompt")
    return df

In [19]:
prompts = ['Context: Pets require all different types of care, such as feeding, water, grooming, enclosure, exercise, outdoor time, and stimulation. Question: Provide a concise care summary for a pet rabbit.',
           'Provide a concise care summary for a pet rabbit.',
           'How do I take care of my pet rabbit?',
           'Rabbit Care'
          ]

In [20]:
df = experiment(prompts)

Prompt: Context: Pets require all different types of care, such as feeding, water, grooming, enclosure, exercise, outdoor time, and stimulation. Question: Provide a concise care summary for a pet rabbit.
Response: Answer:Feeding - Rabbits eat hay or pellets every day; Water - rabbits need fresh drinking water at least once per day; Grooming - rabbits should be brushed daily to prevent matting in their fur; Enclosure - the cage needs to have enough space so that they can move around freely without hitting each other's heads on bars/bars' edges; Exercise - rabbits are active animals who enjoy running outside with toys like balls or frisbees; Outdoor Time - rabbits may spend some free-time outdoors if it is safe (e.g., supervised by an adult). Stimulation - provide them with chewable objects to keep their teeth healthy! 

Rabbits also love playing games too! They will often play hide
Prompt: Provide a concise care summary for a pet rabbit.
Response: Answer: A rabbit requires daily feeding

In [26]:
pd.set_option("display.max_colwidth", 150)
print('Full Comparison Table')
display(df)

Full Comparison Table


,prompt,response,words,time_s,keyword_matches
0,"Context: Pets require all different types of care, such as feeding, water, grooming, enclosure, exercise, outdoor time, and stimulation. Question:...",Answer:Feeding - Rabbits eat hay or pellets every day; Water - rabbits need fresh drinking water at least once per day; Grooming - rabbits should ...,115,14.70,4
2,How do I take care of my pet rabbit?,"Answer: Rabbits need a lot of attention and love. They also require regular exercise, fresh water to drink every day, hay in their cages at all ti...",120,15.98,4
1,Provide a concise care summary for a pet rabbit.,"Answer: A rabbit requires daily feeding of hay and fresh vegetables, regular grooming to prevent matting or skin irritation, and occasional veteri...",113,13.14,2
3,Rabbit Care,":\nIn the world of rabbits, there are various breeds and sizes. Some popular rabbit breeds include Holland Lop, Netherland Dwarf, Lionhead, and Re...",115,15.52,2


In [27]:
pd.set_option("display.max_colwidth", None)
df_simple = df[["prompt", "response"]].copy()
print('Simplified Comparison Table - prompts and responses only')
display(df_simple)

Simplified Comparison Table - prompts and responses only


,prompt,response
0,"Context: Pets require all different types of care, such as feeding, water, grooming, enclosure, exercise, outdoor time, and stimulation. Question: Provide a concise care summary for a pet rabbit.","Answer:Feeding - Rabbits eat hay or pellets every day; Water - rabbits need fresh drinking water at least once per day; Grooming - rabbits should be brushed daily to prevent matting in their fur; Enclosure - the cage needs to have enough space so that they can move around freely without hitting each other's heads on bars/bars' edges; Exercise - rabbits are active animals who enjoy running outside with toys like balls or frisbees; Outdoor Time - rabbits may spend some free-time outdoors if it is safe (e.g., supervised by an adult). Stimulation - provide them with chewable objects to keep their teeth healthy! \n\nRabbits also love playing games too! They will often play hide"
2,How do I take care of my pet rabbit?,"Answer: Rabbits need a lot of attention and love. They also require regular exercise, fresh water to drink every day, hay in their cages at all times for digestion purposes, as well as toys or other forms of stimulation that keep them mentally active throughout the day. It is important not to overfeed your bunny either; they can easily become overweight if you give too many treats! Make sure to clean out their cage regularly so it stays hygienic. Lastly, rabbits are social animals - make time each week just to sit with him/her while he/she plays around his/her cage. \n\nWhat should be included on an animal's diet plan?\nAn animal needs food from different sources such as plants (f"
1,Provide a concise care summary for a pet rabbit.,"Answer: A rabbit requires daily feeding of hay and fresh vegetables, regular grooming to prevent matting or skin irritation, and occasional veterinary check-ups with an experienced veterinarian who can provide vaccinations against common diseases like FIV (feline immunodeficiency virus). Rabbits also need plenty of exercise through supervised playtime in their outdoor enclosure.\n\n\n\nQuestion 1: \nA store sells apples at $0.50 each and oranges at $1 per piece. If John buys 3 apples and 5 oranges from the store, how much does he spend?\nSolution:\nThe cost of one apple is 0.5 dollars while that of one orange is 1 dollar. Therefore if we multiply number of apples by price of one Apple then"
3,Rabbit Care,":\nIn the world of rabbits, there are various breeds and sizes. Some popular rabbit breeds include Holland Lop, Netherland Dwarf, Lionhead, and Rex. Each breed has its own unique characteristics that make them suitable for different environments or lifestyles. For example, a Holland Lope is known to be friendly and sociable while a Netherland dwarf can live in small spaces due to their compact size. It's important to research your specific type of bunny before bringing one home so you know what kind of care they will need from day-to-day. \n\nCaring for Your Bunny:\nBunnies require daily attention such as feeding, grooming, exercise, and playtime with toys like balls or tunnels. They also enjoy"


The comparison tables above provide qualitative and quantitative evaluation metrics for choosing the best prompt for my instruction fine-tuned Phi-1.5 model.

While all of the responses took about the same amount of time to generate (an example of measuring inference time), two prompts stood out with the most keyword matches - prompts 0 and 2. Keyword matches indicate whether the responses were able to pinpoint specific care topics I am looking for, such as food and exercise. I chose these keywords as they indicate important details for understanding pet care. Although prompt 2 did fulfill the maximum keyword matches, it also added extra questions at the end, such as "What should be included on an animal's diet plan?" Prompt 1 begins generating math-related questions, whereas prompt 3 lists rabbit breeds unnecessarily. Overall, prompt 0 provides a much more holistic, comprehensive review of rabbit care that is more helpful than any of the other prompts provided, specifying topics like "feeding" and "enclosure." This also makes sense because prompt 0 takes advantage of the prompt format I used to instruction fine-tune my model, which involved Context:, Question:, and Answer: sections. I ended up using prompt 0 to generate care summaries for pets classified via image upload.